In [8]:
import sys
import os

sys.path.append('SVG/')
sys.path.append('SVG/autoencoder')

In [10]:
from ldm.util import instantiate_from_config
from omegaconf import OmegaConf


In [31]:
config = OmegaConf.load("svg_encoder.yaml")


In [32]:
model = instantiate_from_config(config.model)

Working with z of shape (1, 392, 16, 16) = 100352 dimensions.
making attention of type 'vanilla' with 512 in_channels
making attention of type 'vanilla' with 512 in_channels
making attention of type 'vanilla' with 512 in_channels
making attention of type 'vanilla' with 512 in_channels
Restored from svg_autoencoder_dinov3s16p_vit-s_epoch40.ckpt


In [33]:
model

DinoDecoder(
  (decoder): Decoder(
    (conv_in): Conv2d(392, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (mid): Module(
      (block_1): ResnetBlock(
        (norm1): GroupNorm(32, 512, eps=1e-06, affine=True)
        (conv1): Conv2d(512, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (norm2): GroupNorm(32, 512, eps=1e-06, affine=True)
        (dropout): Dropout(p=0.0, inplace=False)
        (conv2): Conv2d(512, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      )
      (attn_1): AttnBlock(
        (norm): GroupNorm(32, 512, eps=1e-06, affine=True)
        (q): Conv2d(512, 512, kernel_size=(1, 1), stride=(1, 1))
        (k): Conv2d(512, 512, kernel_size=(1, 1), stride=(1, 1))
        (v): Conv2d(512, 512, kernel_size=(1, 1), stride=(1, 1))
        (proj_out): Conv2d(512, 512, kernel_size=(1, 1), stride=(1, 1))
      )
      (block_2): ResnetBlock(
        (norm1): GroupNorm(32, 512, eps=1e-06, affine=True)
        (conv1): Conv2d(512, 512, k

In [34]:
data = instantiate_from_config(config.data)

In [36]:
for x in data.test_dataloader():
    break

x

MisconfigurationException: `test_dataloader` must be implemented to be used with the Lightning Trainer

In [28]:
data.__dir__()

['_log_hyperparams',
 'prepare_data_per_node',
 'allow_zero_length_dataloader_with_multiple_devices',
 'trainer',
 'batch_size',
 'dataset_configs',
 'num_workers',
 'use_worker_init_fn',
 'train_dataloader',
 'val_dataloader',
 'wrap',
 '__module__',
 '__init__',
 'prepare_data',
 'setup',
 '_train_dataloader',
 '_val_dataloader',
 '_test_dataloader',
 '_predict_dataloader',
 '__doc__',
 '__annotations__',
 'name',
 'CHECKPOINT_HYPER_PARAMS_KEY',
 'CHECKPOINT_HYPER_PARAMS_NAME',
 'CHECKPOINT_HYPER_PARAMS_TYPE',
 'from_datasets',
 'state_dict',
 'load_state_dict',
 'on_exception',
 'load_from_checkpoint',
 '__str__',
 'teardown',
 'test_dataloader',
 'predict_dataloader',
 'transfer_batch_to_device',
 'on_before_batch_transfer',
 'on_after_batch_transfer',
 '__dict__',
 '__weakref__',
 '__new__',
 '__repr__',
 '__hash__',
 '__getattribute__',
 '__setattr__',
 '__delattr__',
 '__lt__',
 '__le__',
 '__eq__',
 '__ne__',
 '__gt__',
 '__ge__',
 '__reduce_ex__',
 '__reduce__',
 '__getstate__

In [37]:
data.setup()

ModuleNotFoundError: No module named 'taming.data'

In [ ]:
# Since only_decoder=True, we need to load DINOv3 encoder separately
# Let's load DINOv3 properly
import torch

def load_dinov3_encoder(checkpoint_path, dinov3_repo_path=None):
    """
    Load DINOv3 encoder from checkpoint.
    DINOv3 ViT-S/16+ model loading.
    
    Args:
        checkpoint_path: Path to DINOv3 checkpoint (.pth file)
        dinov3_repo_path: Path to cloned DINOv3 repository (optional)
                          If None, will try to find it or use torch.hub
    """
    print(f"Loading DINOv3 from checkpoint: {checkpoint_path}")
    
    # Try to find DINOv3 repository
    if dinov3_repo_path is None:
        # Check common locations
        possible_paths = [
            'dinov3',
            '../dinov3',
            '../../dinov3',
            os.path.expanduser('~/dinov3'),
        ]
        for path in possible_paths:
            if os.path.exists(path) and os.path.isdir(path):
                dinov3_repo_path = os.path.abspath(path)
                print(f"Found DINOv3 repository at: {dinov3_repo_path}")
                break
    
    # Load checkpoint (it's a direct OrderedDict)
    print(f"Loading checkpoint from {checkpoint_path}...")
    checkpoint = torch.load(checkpoint_path, map_location='cpu')
    
    # The checkpoint is a direct state_dict (OrderedDict)
    state_dict = checkpoint
    
    # Remove 'module.' prefix if present (from DataParallel)
    state_dict = {k.replace('module.', ''): v for k, v in state_dict.items()}
    
    # Load DINOv3 architecture
    if dinov3_repo_path and os.path.exists(dinov3_repo_path):
        # Load from local DINOv3 repository (preferred method)
        print(f"Loading DINOv3 architecture from local repository: {dinov3_repo_path}")
        try:
            encoder = torch.hub.load(
                repo_or_dir=dinov3_repo_path,
                model='dinov3_vits16plus',
                source="local",
                pretrained=False,  # We'll load weights separately
            )
            print("DINOv3 architecture loaded from local repository")
            
            # Load checkpoint weights
            encoder.load_state_dict(state_dict, strict=False)
            print("✓ DINOv3 weights loaded successfully!")
            
        except Exception as e:
            print(f"Error loading from local repo: {e}")
            print("Trying alternative method...")
            raise
    else:
        # Alternative: Try loading from torch.hub (might not work for DINOv3)
        print("DINOv3 repository not found. Trying torch.hub...")
        print("NOTE: DINOv3 requires the official repository.")
        print("Please clone it: git clone https://github.com/facebookresearch/dinov3.git")
        raise FileNotFoundError(
            "DINOv3 repository not found!\n"
            "Please clone the DINOv3 repository:\n"
            "  git clone https://github.com/facebookresearch/dinov3.git\n"
            "Then either:\n"
            "  1. Place it in the current directory as 'dinov3/', or\n"
            "  2. Pass dinov3_repo_path parameter to load_dinov3_encoder()"
        )
    
    encoder.eval()
    return encoder

# Load DINOv3 encoder
dinov3_checkpoint = "dinov3_vits16plus_pretrain_lvd1689m.pth"
if os.path.exists(dinov3_checkpoint):
    try:
        encoder = load_dinov3_encoder(dinov3_checkpoint)
        print("✓ DINOv3 encoder loaded successfully!")
    except FileNotFoundError as e:
        print("\n" + "="*60)
        print("SETUP REQUIRED:")
        print("="*60)
        print(str(e))
        print("="*60)
        raise
else:
    raise FileNotFoundError(
        f"DINOv3 checkpoint not found: {dinov3_checkpoint}\n"
        f"Please ensure the checkpoint file is in the current directory."
    )

In [ ]:
# Now let's process images from imagenet-10 folder
from PIL import Image
import numpy as np
from pathlib import Path
import torch.nn.functional as F

def preprocess_image_for_dino(image_path, size=256):
    """Preprocess image for DINO encoder (same as DinoDecoder.get_input)"""
    img = Image.open(image_path).convert('RGB')
    img = img.resize((size, size), Image.BICUBIC)
    
    # Convert to tensor [0, 1]
    img_tensor = torch.from_numpy(np.array(img)).float() / 255.0
    img_tensor = img_tensor.permute(2, 0, 1)  # HWC -> CHW
    
    # Normalize with ImageNet stats
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    img_tensor = (img_tensor - mean) / std
    
    return img_tensor.unsqueeze(0)  # Add batch dimension

def extract_features(encoder, image_path, device='cuda' if torch.cuda.is_available() else 'cpu'):
    """Extract features from image using DINO encoder"""
    encoder = encoder.to(device)
    img_tensor = preprocess_image_for_dino(image_path).to(device)
    
    with torch.no_grad():
        features = encoder.forward_features(img_tensor)
        
        # Get patch tokens (same as DinoDecoder.encode)
        if isinstance(features, dict):
            if 'x_norm_patchtokens' in features:
                h = features['x_norm_patchtokens']  # [B, N, D]
            elif 'x_prenorm' in features:
                h = features['x_prenorm']
            else:
                h = list(features.values())[0]
        else:
            h = features
        
        # Convert to [B, D, N] if needed, then reshape to [B, D, H, W]
        if h.dim() == 3:
            B, N, D = h.shape
            H_patch = W_patch = int(np.sqrt(N))
            # Reshape to [B, D, H, W] format
            h = h.permute(0, 2, 1).view(B, D, H_patch, W_patch)
    
    return h.cpu()

print("Helper functions defined!")

In [ ]:
# Process all images in imagenet-10 folder
image_folder = Path("imagenet-10")
image_extensions = {'.jpg', '.jpeg', '.png', '.JPEG', '.JPG', '.PNG'}

# Find all images
image_paths = []
for ext in image_extensions:
    image_paths.extend(image_folder.rglob(f'*{ext}'))

print(f"Found {len(image_paths)} images")

# Extract features for all images
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

features_dict = {}
for img_path in image_paths[:5]:  # Process first 5 images as example
    try:
        features = extract_features(encoder, str(img_path), device)
        rel_path = img_path.relative_to(image_folder)
        features_dict[str(rel_path)] = features.squeeze(0)  # Remove batch dim
        print(f"Processed: {rel_path}, features shape: {features.squeeze(0).shape}")
    except Exception as e:
        print(f"Error processing {img_path}: {e}")

print(f"\nSuccessfully processed {len(features_dict)} images")

In [ ]:
# Process all images (not just first 5)
features_dict_all = {}

for img_path in image_paths:
    try:
        features = extract_features(encoder, str(img_path), device)
        rel_path = img_path.relative_to(image_folder)
        features_dict_all[str(rel_path)] = features.squeeze(0)
    except Exception as e:
        print(f"Error processing {img_path}: {e}")

print(f"Total images processed: {len(features_dict_all)}")

# Save features
torch.save(features_dict_all, "svg_encoder_features.pt")
print("Features saved to svg_encoder_features.pt")

# Display sample feature
if features_dict_all:
    sample_path = list(features_dict_all.keys())[0]
    sample_feat = features_dict_all[sample_path]
    print(f"\nSample feature from {sample_path}:")
    print(f"  Shape: {sample_feat.shape}")
    print(f"  Dtype: {sample_feat.dtype}")
    print(f"  Min: {sample_feat.min():.4f}, Max: {sample_feat.max():.4f}, Mean: {sample_feat.mean():.4f}")

In [38]:
model(None)

AttributeError: 'DinoDecoder' object has no attribute 'encoder'